**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Hardware-Accelerated Scientific Computing

> ⚠️ **Draft — requires an NVIDIA GPU; code cells not executed here.** Authored on a machine without CUDA. Run on a CUDA machine or Colab GPU runtime; an instructor should verify each cell before teaching. Remove this banner after that pass.

The sequel [Intro to GPU Systems](./Intro_GPU.ipynb) promised — twice. There you made the GPU *work*; here you make it *earn its keep*: warps and occupancy, shared memory, and streams — the three levers behind every 10× kernel optimization, organized around the CGMA ratio that notebook introduced.

## 1. Pre-requisites

[Intro to GPU Systems](./Intro_GPU.ipynb) (Numba kernels, host/device model). Same conda environment.

In [ ]:
import numpy as np
from numba import cuda
import math, time
print(cuda.detect())        # confirm your device before proceeding

---
### 🕐 Session 1 of 3 — *Warps, Divergence & Occupancy* (~40 min)
**Goal:** understand the 32-thread execution unit and how divergence/occupancy shape speed.
**Builds on:** [Intro to GPU Systems](./Intro_GPU.ipynb). &nbsp; **Feeds into:** Session 2 (shared memory).

---

## 2. The Warp

💡 **Intuition.** The GPU does not execute threads — it executes **warps**: teams of 32 threads in lockstep, one instruction for all. Two consequences rule kernel design. **Divergence:** an `if` that splits a warp forces both paths to run serially (idle threads masked) — branch on *warp-aligned* conditions when you can. **Occupancy:** each SM juggles many warps and swaps them zero-cost whenever one stalls on memory; enough resident warps = latency *hidden*. Registers and shared memory per block cap how many fit — the resource budget behind every launch config.

In [ ]:
@cuda.jit
def diverge_bad(x, out):
    i = cuda.grid(1)
    if i < x.size:
        if i % 2 == 0:                      # even/odd splits EVERY warp
            out[i] = math.sin(x[i])
        else:
            out[i] = math.cos(x[i])

@cuda.jit
def diverge_good(x, out):
    i = cuda.grid(1)
    if i < x.size:
        if (i // 32) % 2 == 0:              # whole warps take the same path
            out[i] = math.sin(x[i])
        else:
            out[i] = math.cos(x[i])

x_d = cuda.to_device(np.random.rand(2**24).astype(np.float32))
out_d = cuda.device_array_like(x_d)
for name, k in [("warp-splitting", diverge_bad), ("warp-aligned", diverge_good)]:
    k[x_d.size // 256 + 1, 256](x_d, out_d); cuda.synchronize()      # warm up / JIT
    tic = time.perf_counter()
    for _ in range(50): k[x_d.size // 256 + 1, 256](x_d, out_d)
    cuda.synchronize()
    print(f"{name:15s}: {(time.perf_counter()-tic)/50*1e3:.2f} ms")

---
### 🕐 Session 2 of 3 — *Shared Memory & Tiling* (~40 min)
**Goal:** raise CGMA by staging data in on-chip memory; tile a matrix multiply.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (streams).

---

## 3. The On-Chip Scratchpad

💡 **Intuition.** Global memory is the slow warehouse; **shared memory** is a small, fast workbench each block controls explicitly. The tiling pattern: every thread of a block cooperatively loads a tile of the operands into shared memory (each element fetched from global memory **once**), synchronizes, then all threads reuse the tile many times. Reuse is exactly the CGMA ratio [Intro to GPU Systems §4](./Intro_GPU.ipynb) defined — tiling is how you buy it.

In [ ]:
TILE = 16
@cuda.jit
def matmul_tiled(A, B, C):
    sA = cuda.shared.array((TILE, TILE), dtype=np.float32)
    sB = cuda.shared.array((TILE, TILE), dtype=np.float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    row, col = cuda.grid(2)
    acc = 0.0
    for t in range((A.shape[1] + TILE - 1) // TILE):
        if row < A.shape[0] and t*TILE + tx < A.shape[1]:
            sA[ty, tx] = A[row, t*TILE + tx]
        else: sA[ty, tx] = 0.0
        if col < B.shape[1] and t*TILE + ty < B.shape[0]:
            sB[ty, tx] = B[t*TILE + ty, col]
        else: sB[ty, tx] = 0.0
        cuda.syncthreads()                      # tile fully loaded before anyone computes
        for k in range(TILE):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()                      # everyone done before the tile is overwritten
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = acc

N = 1024
A = cuda.to_device(np.random.rand(N, N).astype(np.float32))
B = cuda.to_device(np.random.rand(N, N).astype(np.float32))
C = cuda.device_array((N, N), np.float32)
grid = ((N+TILE-1)//TILE, (N+TILE-1)//TILE)
matmul_tiled[grid, (TILE, TILE)](A, B, C); cuda.synchronize()
tic = time.perf_counter(); matmul_tiled[grid, (TILE, TILE)](A, B, C); cuda.synchronize()
t_tiled = time.perf_counter() - tic
print(f"tiled matmul: {t_tiled*1e3:.1f} ms  ≈ {2*N**3/t_tiled/1e9:.0f} GFLOP/s")
print("compare against a naive (untiled) kernel and against cupy's cuBLAS call — the ladder is the lesson")

---
### 🕐 Session 3 of 3 — *Streams & Overlap* (~35 min)
**Goal:** overlap transfers with compute; never let either engine idle.
**Builds on:** Session 2.

---

## 4. Streams

💡 **Intuition.** The GPU has separate engines for copying and computing — and by default you use them one at a time. **Streams** are independent work queues: chunk the data, and while chunk $k$ computes, chunk $k{+}1$ uploads and chunk $k{-}1$ downloads. Perfect overlap hides the smaller of transfer/compute time entirely — the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) pipeline idea, on silicon. Requires *pinned* host memory (`cuda.pinned_array`) so DMA can run without the [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) paging underneath it.

In [ ]:
n_chunks, chunk = 8, 2**22
streams = [cuda.stream() for _ in range(n_chunks)]
host = [cuda.pinned_array(chunk, np.float32) for _ in range(n_chunks)]
for h in host: h[:] = np.random.rand(chunk)

@cuda.jit
def heavy(x):
    i = cuda.grid(1)
    if i < x.size:
        v = x[i]
        for _ in range(60): v = math.sin(v) * 1.0001
        x[i] = v

# serial: one stream does copy→compute→copy for each chunk in turn
tic = time.perf_counter()
for h in host:
    d = cuda.to_device(h)
    heavy[chunk//256+1, 256](d)
    d.copy_to_host(h)
cuda.synchronize(); t_serial = time.perf_counter() - tic

# overlapped: each chunk on its own stream
tic = time.perf_counter()
devs = []
for h, s in zip(host, streams):
    d = cuda.to_device(h, stream=s)
    heavy[chunk//256+1, 256, s](d)
    d.copy_to_host(h, stream=s)
cuda.synchronize(); t_stream = time.perf_counter() - tic
print(f"serial {t_serial*1e3:.0f} ms   streamed {t_stream*1e3:.0f} ms   overlap bought {t_serial/t_stream:.2f}x")

## 5. Conclusion

Think in warps (divergence), feed from the workbench (shared memory raises CGMA), and keep both engines busy (streams). Profile with Nsight before and after each change — the tool tells you which lever is binding.

---
## Where next

- [CUDA in C++](./CUDA_Cpp.ipynb) — the same levers without the Python training wheels.
- [Scaling Neural Networks](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — these ideas at training scale.